# 1D J1J2 Inference: GRU variants (Euclidean, Poincare, Lorentz)

This notebook is part of the work arXiv:2604.24337 [quant-ph] (https://arxiv.org/html/2604.24337v1). In this notebook, I performed the inference process for the trained neural quantum states using 10000 samples. The loading directory in the Github repo might differ from the loading path used here. Please check and use the correct loading path. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('utility_lorentz')
from j1j2_train_loop_lorentz import *

sys.path.append('utility_poincare')
from j1j2_hyprnn_train_loop import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False
GPU Available:  False


In [6]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))    
    if mad == 0:
        return eloc       
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad    
    # Clip the values (keeping the imaginary part if it exists)
    # Create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)   
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test(wf, numsamples, path_to_weights, Ee, clipped_e = False):
    if 'Lorentz' in wf.name:
        test_samples_before = wf.sample_no_tau(numsamples)
    else: 
        test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_before, Marshall_sign=True)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')
     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the re_mapped weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")   
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        if 'Lorentz' in wf.name:
            test_samples_after = wf.sample_no_tau(numsamples)
        else: 
            test_samples_after = wf.sample(numsamples)               
        if clipped_e:
            # Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)  
            # Clipping
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)   
            # Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)    
            # Count how many were clipped 
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fcn = 'trained_nqs/J1J2'

## J2=0.0

In [4]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251

In [6]:
# EuclGRU: -43.4337 (with clipped E) --> Update using this
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.352399826049805-0.009700000286102295j), var E = 0.5076000094413757
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.414100646972656+0.003100000089034438j), var E = 1.166200041770935
DMRG energy  is -44.1277
Time taken=0.086 hrs


In [7]:
# PoincareGRU 
# with clipped E (764 outliers): Mean E = (-43.12699890136719+0.005100000184029341j), var E = 0.8920000195503235
rmax=1.0
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_00, clipped_E=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.481599807739258-0.010599999688565731j), var E = 0.491100013256073
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.0526008605957+0.005100000184029341j), var E = 3.7451999187469482
DMRG energy  is -44.1277
Time taken=0.348 hrs


In [9]:
#LorentzGRU: spatial_clamp = 4.0: -43.8034
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2=0.0_sc=4.0/N100_J1=1.0|J2={J2_}_100_LorentzGRU_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (24.723499298095703-0.0010999999940395355j), var E = 0.18490000069141388
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.80339813232422+0.0015999999595806003j), var E = 0.3580999970436096
DMRG energy  is -44.1277
Time taken=1.015 hrs


## J2=0.2

In [21]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388

In [10]:
# EuclGRU
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (29.063600540161133-0.011900000274181366j), var E = 0.8237000107765198
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.945098876953125-0.0008999999845400453j), var E = 0.5217999815940857
DMRG energy  is -40.7388
Time taken=0.12 hrs


In [11]:
# PoincareGRU
# without clipped E: Mean E = (-40.407901763916016-0.0010000000474974513j), var E = 0.3495999872684479
rmax=1.0
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (29.257299423217773-0.014800000004470348j), var E = 0.8708000183105469
Successfully remapped and loaded weights.
Clipped 653 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.421600341796875-0.0010000000474974513j), var E = 0.21969999372959137
DMRG energy  is -40.7388
Time taken=0.424 hrs


In [22]:
#LorentzGRU: spatial_clamp = 6.0
# without clipped E: Mean E = (-39.95750045776367-0.0026000000070780516j), var E = 0.6586999893188477
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=6.0, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2=0.2_sc=6.0/N100_J1=1.0|J2={J2_}_100_LorentzGRU_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (28.50480079650879+0.014800000004470348j), var E = 1.7141000032424927
Successfully remapped and loaded weights.
Clipped 564 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.035400390625-0.0026000000070780516j), var E = 0.27730000019073486
DMRG energy  is -40.7388
Time taken=1.57 hrs


## J2=0.5

In [13]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000

In [14]:
# EuclGRU
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.13029861450195-0.015200000256299973j), var E = 1.576200008392334
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.80619812011719+0.0017999999690800905j), var E = 0.39879998564720154
DMRG energy  is 37.5
Time taken=0.075 hrs


In [15]:
# PoincareGRU
rmax=1.0
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.4208984375-0.0210999995470047j), var E = 1.7258000373840332
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.459999084472656+0.0017999999690800905j), var E = 0.0771000012755394
DMRG energy  is 37.5
Time taken=0.498 hrs


In [16]:
#LorentzGRU
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=6.0, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2=0.5_sc=4.0/N100_J1=1.0|J2={J2_}_100_LorentzGRU_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.91270065307617-0j), var E = 0.46540001034736633
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.47629928588867-0.0012000000569969416j), var E = 0.054499998688697815
DMRG energy  is 37.5
Time taken=1.469 hrs


## J2=0.8

In [17]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643

In [18]:
# EuclGRU
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (43.196998596191406-0.01850000023841858j), var E = 2.6628000736236572
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.881099700927734-0.011099999770522118j), var E = 1.5683000087738037
DMRG energy  is -42.0701
Time taken=0.076 hrs


In [19]:
# PoincareGRU
rmax=1.0
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (43.58449935913086-0.027400000020861626j), var E = 2.9233999252319336
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.465301513671875+0.0010000000474974513j), var E = 0.37630000710487366
DMRG energy  is -42.0701
Time taken=0.538 hrs


In [20]:
#LorentzGRU
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=6.0, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2=0.8_sc=6.0/N100_J1=1.0|J2={J2_}_100_LorentzGRU_{units}_ns=80_MsTrue_checkpoint.pt'
# The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (44.226200103759766+0.0006000000284984708j), var E = 0.7383000254631042
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.54880142211914+0.005100000184029341j), var E = 1.9919999837875366
DMRG energy  is -42.0701
Time taken=2.202 hrs
